# 02 - Demand forecasting (run on Google Colab)

Trains and backtests five models (seasonal naive, moving average, SARIMA with Fourier terms, Prophet,
global LightGBM) on weekly units per category, picks a champion, and writes the 12-week forecast.
All logic lives in `src/sca/forecasting.py` and `src/sca/forecast_report.py`; this notebook only runs it.

**Workflow**
1. Run the setup cells (clone, install).
2. Provide the data: upload `sca_processed.zip` (made locally, see below) **or** use Kaggle.
3. Run the forecast (a few minutes on a standard Colab CPU runtime).
4. Download `forecast_outputs.zip`, unzip it into the repo root on your machine, then commit
   `reports/forecast_results.md` and `reports/figures/forecast_*.png`.

Make `sca_processed.zip` on your machine from the repo root (PowerShell):
```
python -m sca ingest
Compress-Archive -Path data\processed\*.parquet, data\processed\meta.json -DestinationPath sca_processed.zip -Force
```

In [ ]:
# 1. Setup: get the code and install pinned dependencies.
REPO = "https://github.com/swarnkarkuldeep/SupplyPulse.git"  # change if you fork it
!git clone -q $REPO SupplyPulse || echo "already cloned"
%cd SupplyPulse
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
# If the pinned install fails on this Colab Python version, use the unpinned install instead:
# !pip install -q -e .

## 2. Data

**Option A (recommended):** upload `sca_processed.zip`.

In [ ]:
import pathlib
import zipfile

from google.colab import files

up = files.upload()  # choose sca_processed.zip
name = next(iter(up))
pathlib.Path("data/processed").mkdir(parents=True, exist_ok=True)
zipfile.ZipFile(name).extractall("data/processed")
!ls -la data/processed

**Option B:** download from Kaggle and run the ingest here (needs your `kaggle.json`). Skip this cell if you used option A. Check the dataset slug on the Kaggle page if the download fails.

In [ ]:
# files.upload() your kaggle.json first
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d shashwatwork/dataco-smart-supply-chain-for-big-data-analysis -p data/raw --unzip
# !python -m sca ingest

## 3. Run the forecast

Uses 2 parallel workers for the per-series models. Takes a few minutes.

In [ ]:
import yaml

cfg = yaml.safe_load(open("config.yaml"))
cfg["forecast"]["n_jobs"] = 2
yaml.safe_dump(cfg, open("colab_config.yaml", "w"))
!python -m sca forecast --config colab_config.yaml

## 4. Results

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

out = "data/processed/forecast"
display(pd.read_csv(f"{out}/metrics_overall.csv").round(4))
display(Markdown(open("reports/forecast_results.md").read().split("## Champion")[1].split("## ")[0]))
for fig in ("forecast_01_model_wape", "forecast_02_category_wape", "forecast_03_forecasts"):
    display(Image(f"reports/figures/{fig}.png"))

## 5. Download the outputs

Unzip `forecast_outputs.zip` into the repo root on your machine.

In [ ]:
!zip -qr forecast_outputs.zip reports/forecast_results.md reports/figures/forecast_*.png data/processed/forecast
from google.colab import files

files.download("forecast_outputs.zip")